# 13.2: A worked example: Llama 2 7B and Llama 3 8B

Llama 3 8B has the same $L$ and $d$ but uses grouped-query attention with 8 key-value heads, a wider MLP ($d_{\text{ff}} = 14336$), and a much larger vocabulary (128,256 tokens). Counting exactly is easy in code:


In [ ]:
def param_count(L, d, n_heads, n_kv_heads, d_ff, vocab, tied_embeddings=False, gated_mlp=True):
    d_head = d // n_heads
    attn = 2 * d * d + 2 * d * (n_kv_heads * d_head)       # W_Q, W_O, W_K, W_V
    mlp = (3 if gated_mlp else 2) * d * d_ff
    norms = 2 * d                                            # two RMSNorm weight vectors
    blocks = L * (attn + mlp + norms)
    emb = vocab * d * (1 if tied_embeddings else 2)
    return blocks, emb, blocks + emb

blocks, emb, total = param_count(L=32, d=4096, n_heads=32, n_kv_heads=8, d_ff=14336, vocab=128256)
print(f"blocks {blocks/1e9:.2f}B, embeddings {emb/1e9:.2f}B, total {total/1e9:.2f}B")
# blocks 6.98B, embeddings 1.05B, total 8.03B
